# Кросс-валидация по пациентам

Обучаем базовую модель пять раз на разных группах **тренировочных** пациентов. В каждом фолде все окна одного пациента находятся только в train или только в val. Сохранённые `val` и `test` остаются вне CV. Этот пример проверяет рабочий процесс; качество модели оценивайте по полноценной выборке и финальному отложенному test.

Сначала разметьте каталог и сохраните разбиение через `WindowSet.build()` и `WindowSet.split(seed=42)`, если они ещё не созданы. Настройки окон ниже должны совпадать с настройками разметки. Не пересоздавайте split между экспериментами.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from epi import Catalog, WindowSet
from epi.evaluation import patient_cv, patient_folds
from epi.features import eeg_features
from epi.preprocessing import preprocess

DB_PATH = "catalog.sqlite"  # Для коллекции F укажите "catalog_F.sqlite".
N_WINDOWS = 2000
N_SPLITS = 5
SEED = 42


In [ ]:
catalog = Catalog(DB_PATH)
windows = WindowSet(catalog, length_sec=10.0, stride_sec=5.0, min_overlap_sec=5.0)
if not windows.windows("train", limit=1):
    raise ValueError("Нет окон train: сначала выполните windows.build() и windows.split(seed=SEED)")

x, y, groups = windows.sample(
    "train", n=N_WINDOWS, seed=SEED, return_groups=True,
)
print("Пациентов:", len(np.unique(groups)), "| Окон:", len(y), "| Положительных:", int(y.sum()))


Фильтры с фиксированными параметрами и признаки ниже рассчитываются отдельно для каждого окна. Все преобразования, которым нужен `fit` на нескольких окнах, должны обучаться внутри фолда: здесь `StandardScaler` включён в Pipeline. Если используете `fit_quality_limits`, обучайте его на train текущего фолда. При фильтрации окон применяйте одну маску к `x`, `y` и `groups`.

Новые `.npz` из `windows.export()` уже содержат `groups`. Старые экспорты без ключей пациентов нужно пересоздать.

In [ ]:
features, feature_names = eeg_features(preprocess(x, windows.rate), windows.rate)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
)
result = patient_cv(
    model, features, y, groups, n_splits=N_SPLITS, seed=SEED, verbose=True,
)


In [ ]:
columns = ["fold", "train_patients", "val_patients", "train", "val",
           "train_positive", "val_positive", "roc_auc", "pr_auc"]
display(pd.DataFrame(result["folds"])[columns])
display(pd.DataFrame(result["summary"]).T)


`summary` содержит невзвешенное среднее и стандартное отклонение метрик по фолдам (`ddof=0`), а также число определённых значений `n_valid` из `n_folds`. Если в val один класс, AUC равен `NaN` и в среднее не входит; предупреждение покажет такой фолд. Если один класс остался в train, CV остановится до обучения. При числе пациентов меньше запрошенных фолдов их число сократится с предупреждением.

`overall` — отдельные метрики объединённых out-of-fold предсказаний, они могут отличаться от среднего по фолдам. `std` показывает разброс между группами пациентов и не является доверительным интервалом. При балансированной выборке PR AUC относится к её доле положительных окон.

Для нейросети используйте `patient_folds(groups, N_SPLITS, y=y, seed=SEED)` и создавайте модель и оптимизатор заново в каждом фолде. Метрики такого цикла можно свести через `epi.evaluation.summarize_folds()`.

In [ ]:
# Проверка: каждый пациент и каждое окно побывали в val ровно один раз.
seen = np.zeros(len(y), dtype=int)
for train_idx, val_idx in result["indices"]:
    assert not set(groups[train_idx]) & set(groups[val_idx])
    seen[val_idx] += 1
assert np.all(seen == 1)
print("Пересечений пациентов между train и val нет.")
catalog.close()
